<a href="https://colab.research.google.com/github/sefas100/fact-checking-bge-rag/blob/main/Fact_Checking_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bachelorarbeit: Finetuning eines BGE-Modells für Fact-Checking

## 0. Initiales Setup und Datenzugriff

### Abhängigkeiten installieren

In [1]:
%%capture
!pip uninstall -y torchaudio
!pip install -U --no-deps sentence-transformers
!pip install -U transformers datasets bitsandbytes accelerate google-generativeai xformers

### Setup und Bibliotheksimporte

In [3]:
# SETUP & IMPORTS

import shutil
import os
import gc
import json
import torch
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments, util
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Falls der Pfad blockiert ist, den Ordner bereinigen
mount_path = '/content/drive'
if os.path.exists(mount_path) and os.listdir(mount_path):
    print("Mountpoint ist nicht leer. Leere Pfad...")
    shutil.rmtree(mount_path)

# Jetzt Drive sauber einhängen
drive.mount(mount_path, force_remount=True)
base_path = '/content/drive/MyDrive/BachlorArbeit/'
if os.path.exists(base_path):
    print("Verbindung erfolgreich! Dateien im Ordner:")
    print(os.listdir(base_path))

/tmp/ipykernel_10302/1198684315.py:16: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator
/tmp/ipykernel_10302/1198684315.py:17: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss


Mounted at /content/drive
Verbindung erfolgreich! Dateien im Ordner:
['README.md', 'pairs.csv', 'eng_posts.csv', 'eng_fact.csv', 'Fact_check.ipynb', 'extract_Fact.ipynb', 'eng_pairs_cleaned.csv', 'extract_eng.ipynb', 'baselines', 'Versuche', '.vscode', 'train_dev_sets', 'test_set', 'scripts', '937_bestes_Ergebnis', 'fact_embeddings_base.pt', 'training_data_final.csv', 'negatives_base_data.csv', 'bge_finetuned', 'models', 'bge_train_data.csv', 'bge_training_logs', 'temp_bge_train.csv', 'bge_pro_logs', 'temp_bge_final.csv', 'tensorboard_logs', 'bge_pro_checkpoints', 'evaluation_results.csv', 'final_evaluation_results.csv', 'train.jsonl', 'fever_embeddings.pt', 'fever_embeddings_bge_pro.pt', 'negatives_base_data.gsheet', 'eng_posts.gsheet', 'eng_fact.gsheet', 'matryoshka_evaluation_results.csv', 'corpus_embeddings.pt', 'rag_predictions.csv', 'corpus_embeddings_base.pt', 'corpus_embeddings_pro.pt', 'rag_predictions_comparison.csv', 'rag_predictions_500.csv', 'rag_predictions_300.csv', 'rag

### Datenvorbereitung

In [ ]:
# DATEN VORBEREITUNG
#
# df_posts = pd.read_csv(os.path.join(base_path, 'eng_posts.csv'))
# df_facts = pd.read_csv(os.path.join(base_path, 'eng_fact.csv'))
# df_pairs = pd.read_csv(os.path.join(base_path, 'eng_pairs_cleaned.csv'))
#
# train_df = df_pairs.merge(df_facts[['fact_check_id', 'claim']], on='fact_check_id')
# train_df = train_df.rename(columns={'english_text': 'query', 'claim': 'positive'})
# train_df[['query', 'positive']].to_csv(os.path.join(base_path, 'negatives_base_data.csv'), index=False)
# print("Vorbereitung abgeschlossen. Datei 'negatives_base_data.csv' wurde erstellt.")

### Korpus-Embeddings Generierung

In [ ]:
# #FEVER EMBEDDINGS BERECHNEN (DOKUMENTATION)

# data = []
# with open('/content/drive/MyDrive/train.jsonl', 'r', encoding='utf-8') as f:
#     for line in f:
#         data.append(json.loads(line))
# df_fever = pd.DataFrame(data)

# model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# corpus_embeddings = model.encode(df_fever['claim'].tolist(), show_progress_bar=True, convert_to_tensor=True)
# torch.save(corpus_embeddings, '/content/drive/MyDrive/BachlorArbeit/fever_embeddings.pt')

# # BGE Base für die Evaluation
# import os
# import json
# import torch
# import pandas as pd
# from sentence_transformers import SentenceTransformer

# # 1. SETUP & PFADE
# BASE_PATH = "/content/drive/MyDrive/BachlorArbeit/"
# FEVER_FILE_PATH = os.path.join(BASE_PATH, "train.jsonl")
# MODEL_BASE_NAME = "BAAI/bge-base-en-v1.5"
# OUTPUT_EMBEDDINGS_PATH = os.path.join(BASE_PATH, "fever_embeddings_bge_base.pt")

# # 2. FEVER-DATEN LADEN
# print("Lade FEVER-Korpus aus train.jsonl...")
# data = []
# with open(FEVER_FILE_PATH, 'r', encoding='utf-8') as f:
#     for line in f:
#         data.append(json.loads(line))

# df_fever = pd.DataFrame(data)
# claims_list = df_fever['claim'].tolist()
# print(f"Erfolgreich {len(claims_list)} Claims geladen.")

# # 3. BGE-BASE MODELL LADEN & EMBEDDINGS BERECHNEN
# print(f"\nLade Basis-Modell '{MODEL_BASE_NAME}'...")
# model_base = SentenceTransformer(MODEL_BASE_NAME).cuda()

# print("Starte Berechnung der Embeddings für BGE-Base (das kann einige Minuten dauern)...")
# corpus_embeddings_base = model_base.encode(
#     claims_list,
#     show_progress_bar=True,
#     convert_to_tensor=True
# )

# # 4. SPEICHERN
# torch.save(corpus_embeddings_base, OUTPUT_EMBEDDINGS_PATH)
# print(f"\nFertig! Embeddings erfolgreich gespeichert unter:\n{OUTPUT_EMBEDDINGS_PATH}")

### Trainingsdaten-Visualisierung

In [ ]:
# MODELL TRAINING (BGE-PRO MIT MATRYOSHKA)
#
# model_id = "BAAI/bge-base-en-v1.5"
# output_model_path = os.path.join(base_path, "models/bge_final_ba_pro")
# log_dir = os.path.join(base_path, "tensorboard_logs")
# data_path = os.path.join(base_path, "negatives_base_data.csv")
#
# query_instruction = "Represent this sentence for searching relevant passages: "
# os.makedirs(output_model_path, exist_ok=True)
# os.makedirs(log_dir, exist_ok=True)
#
# df = pd.read_csv(data_path).dropna()
# df['query_with_instruction'] = query_instruction + df['query'].astype(str)
# temp_csv = os.path.join(base_path, "temp_bge_pro.csv")
# df.to_csv(temp_csv, index=False)
#
# raw_dataset = load_dataset("csv", data_files=temp_csv, split="train")
# dataset_with_negatives = util.mine_hard_negatives(
#     dataset=raw_dataset,
#     anchor_column_name="query_with_instruction",
#     positive_column_name="positive",
#     model=model_id,
#     num_negatives=3,
#     batch_size=128
# )
# train_test_split = dataset_with_negatives.train_test_split(test_size=0.1)
#
# model = SentenceTransformer(model_id).cuda()
# inner_loss = MultipleNegativesRankingLoss(model=model)
# train_loss = MatryoshkaLoss(
#     model=model,
#     loss=inner_loss,
#     matryoshka_dims=[64, 128, 256, 512, 768]
# )
#
# training_args = SentenceTransformerTrainingArguments(
#     output_dir=output_model_path,
#     per_device_train_batch_size=32,
#     learning_rate=1e-5,
#     bf16=True,
#     save_total_limit=1,
#     report_to="tensorboard",
#     logging_dir=log_dir,
#     logging_steps=10,
#     eval_strategy="steps",
#     eval_steps=50
# )
#
# trainer = SentenceTransformerTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_test_split["train"],
#     eval_dataset=train_test_split["test"],
#     loss=train_loss
# )
# trainer.train()
# model.save(output_model_path)
# del model
# gc.collect()
# torch.cuda.empty_cache()


### RAG Pipeline Implementierung

In [ ]:
# # TRAININGSDATEN-VISUALISIERUNG
# # Echte Daten aus dem Training-Log
# steps = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800]
# train_loss = [5.60, 5.09, 4.16, 4.38, 4.17, 2.82, 2.61, 2.07, 2.11, 2.32, 2.49, 1.75, 2.39, 1.56, 2.10, 1.63]
# val_loss = [3.69, 3.13, 2.73, 2.42, 2.14, 1.93, 1.71, 1.57, 1.47, 1.35, 1.27, 1.22, 1.17, 1.13, 1.11, 1.10]

# # Metriken für Basis vs Pro Modell
# labels = ['S@1', 'S@3', 'S@5', 'S@10']
# basis_bge = [0.480, 0.724, 0.780, 0.812]
# finetuned_pro = [0.620, 0.888, 0.948, 0.964]

# # Matryoshka Daten
# dims = [64, 128, 256, 512, 768]
# matry_s1 = [0.620, 0.656, 0.652, 0.652, 0.620]

# # Dashboard Erstellen
# sns.set_theme(style="whitegrid")
# fig, axes = plt.subplots(2, 2, figsize=(18, 12))
# plt.subplots_adjust(hspace=0.3, wspace=0.2)

# # Trainings- vs Validierungs-Loss
# axes[0, 0].plot(steps, train_loss, label='Trainings-Loss', color='#cfd8dc', linestyle='--', alpha=0.8)
# axes[0, 0].plot(steps, val_loss, label='Validierungs-Loss', color='#1a5f7a', linewidth=3, marker='o')
# axes[0, 0].set_title('A: Lernkurve (Loss Verlauf)', fontsize=14, fontweight='bold')
# axes[0, 0].set_xlabel('Trainings-Schritte')
# axes[0, 0].set_ylabel('Loss Wert')
# axes[0, 0].legend()

# # Benchmark-Vergleich
# x = np.arange(len(labels))
# width = 0.35
# axes[0, 1].bar(x - width/2, basis_bge, width, label='Basis BGE', color='#cfd8dc')
# axes[0, 1].bar(x + width/2, finetuned_pro, width, label='BGE Pro (Finetuned)', color='#1a5f7a')
# axes[0, 1].set_title('B: Performance-Steigerung (S@k)', fontsize=14, fontweight='bold')
# axes[0, 1].set_xticks(x)
# axes[0, 1].set_xticklabels(labels)
# axes[0, 1].set_ylim(0, 1.1)
# axes[0, 1].legend()

# for i in range(len(labels)):
#     axes[0, 1].text(i - width/2, basis_bge[i] + 0.02, f"{basis_bge[i]:.2f}", ha='center', fontsize=10)
#     axes[0, 1].text(i + width/2, finetuned_pro[i] + 0.02, f"{finetuned_pro[i]:.2f}", ha='center', fontweight='bold')

# # Matryoshka-Dimensions-Analyse
# axes[1, 0].plot(dims, matry_s1, color='#d9534f', linewidth=3, marker='o', markersize=10)
# axes[1, 0].set_title('C: Matryoshka-Effizienz (S@1)', fontsize=14, fontweight='bold')
# axes[1, 0].set_xlabel('Vektor-Dimension')
# axes[1, 0].set_ylabel('Genauigkeit')
# axes[1, 0].set_ylim(0.5, 0.7)

# # Recall-Sicherheit für den Bot
# sns.barplot(x=labels, y=finetuned_pro, ax=axes[1, 1], hue=labels, palette='viridis', legend=False)
# axes[1, 1].set_title('D: Sicherheit der Fakten-Suche (Recall)', fontsize=14, fontweight='bold')
# axes[1, 1].set_ylabel('Wahrscheinlichkeit')
# axes[1, 1].set_ylim(0, 1.1)

# for i, v in enumerate(finetuned_pro):
#     axes[1, 1].text(i, v + 0.02, f"{v*100:.1f}%", ha='center', fontweight='bold')

# plt.savefig("Bachelorarbeit_Final_Dashboard.png", dpi=300, bbox_inches='tight')
# plt.show()

### RAG Pipeline Testfälle

In [ ]:
# KONFIGURATION
FILE_PATH = '/content/drive/MyDrive/BachlorArbeit/train.jsonl'
MODEL_PATH = '/content/drive/MyDrive/BachlorArbeit/models/bge_final_ba_pro'
EMBEDDINGS_PATH = '/content/drive/MyDrive/BachlorArbeit/fever_embeddings_bge_pro.pt'
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "

# Verwende den exakten Hugging-Face-Pfad des Qwen-Modells (z.B. "Qwen/Qwen2.5-7B-Instruct")
LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# INIZIALISIERUNG DES LOKALEN QWEN-MODELLS
print("--- Loading Local Qwen Language Model ---")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Erstellen einer lokalen Pipeline für die Textgenerierung
local_generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer
)

# LADEN DES FINETUNED RETRIEVAL SYSTEMS
print("\n--- Initializing Finetuned Retrieval System ---")
if not os.path.exists(MODEL_PATH):
    print(f"FEHLER: Dein trainiertes Modell wurde unter {MODEL_PATH} nicht gefunden!")
else:
    search_model = SentenceTransformer(MODEL_PATH).cuda()

    # Textdaten laden
    data = []
    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    df_fever = pd.DataFrame(data)
    print(f"Success: {len(df_fever)} claims loaded.")

    # Embeddings laden oder neu generieren
    if os.path.exists(EMBEDDINGS_PATH):
        print("Loading pre-computed embeddings for your Finetuned Model...")
        corpus_embeddings = torch.load(EMBEDDINGS_PATH)
    else:
        print("First Run with new Model: Generating index...")
        corpus_embeddings = search_model.encode(df_fever['claim'].tolist(), convert_to_tensor=True, show_progress_bar=True)
        torch.save(corpus_embeddings, EMBEDDINGS_PATH)
        print(f"Saved new embeddings to {EMBEDDINGS_PATH}")

# LOKALE FACT-CHECK LOGIK
def run_detailed_fact_check(query):
    if 'corpus_embeddings' not in globals():
        print("System not ready.")
        return

    print(f"\nQUERY: {query}")
    print("-" * 60)

    # Retrieval
    full_query = QUERY_INSTRUCTION + query
    query_emb = search_model.encode(full_query, convert_to_tensor=True)
    hits = util.semantic_search(query_emb, corpus_embeddings, top_k=5)[0]

    evidence_context = ""
    for hit in hits:
        if hit['score'] >= 0.30:
            idx = hit['corpus_id']
            row = df_fever.iloc[idx]
            evidence_context += (
                f"- Evidence: {row['claim']} "
                f"(Match Score: {hit['score']:.4f})\n"
            )

    #Lokale Prompt-Strukturierung (Verwendung des Qwen-Chat-Templates)
    messages = [
        {"role": "system", "content": "You are a professional fact-checker. Provide detailed English responses with logical reasoning based on the provided evidence."},
        {"role": "user", "content": f"""The following evidence was retrieved using a custom-tuned BGE model from the FEVER database:
{evidence_context}

User Question: {query}

Task:
1. Provide a verdict: SUPPORTS, REFUTES, or NOT ENOUGH INFO.
2. Explain the reasoning in English, specifically mentioning the match quality of the evidence in %.
3. If the evidence matches strongly (high Match Score), emphasize this."""}
    ]

    # Formatieren des Prompts nach Qwen-Vorgaben
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    print("AI is analyzing locally with Finetuned Intelligence...")

    # Lokale Generierung (Entspricht deiner niedrigen Temperature-Einstellung)
    outputs = local_generator(
        prompt,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.2,
        top_p=0.9
    )

    # Abschneiden des Input-Prompts aus der Ausgabe
    response = outputs[0]["generated_text"][len(prompt):]

    print("\nDETAILED ASSESSMENT:\n" + "="*60 + "\n" + response + "\n" + "="*60)

# Test
if 'corpus_embeddings' in globals():
    run_detailed_fact_check("The Earth is flat.")

--- Loading Local Qwen Language Model ---


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


--- Initializing Finetuned Retrieval System ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Success: 145449 claims loaded.
Loading pre-computed embeddings for your Finetuned Model...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUERY: The Earth is flat.
------------------------------------------------------------
AI is analyzing locally with Finetuned Intelligence...

DETAILED ASSESSMENT:
Verdict: REFUTES

Reasoning:
The provided evidence clearly supports the fact that Earth is not flat but rather a spherical astronomical body within our solar system. Here's a detailed breakdown:

1. **Evidence Quality**: The evidence provided has high match scores, indicating strong alignment with factual information about Earth. Specifically, the evidence states:
   - "Earth is outside of the Solar System" (Match Score: 66.85%)
   - "Earth is an astronomical body" (Match Score: 64.31%)
   - "Earth is a planet" (Match Score: 62.47%)

2. **Logical Reasoning**:
   - The statement "Earth is outside of the Solar System" is incorrect and directly contradicts well-established scientific knowledge. Earth is indeed part of the Solar System, orbiting around the Sun.
   - The statements "Earth is an astronomical body" and "Earth is a

In [ ]:
# 1. Beispiel:
run_detailed_fact_check("Barack Obama was born in Kenya.")

# 2. Beispiel:
run_detailed_fact_check("The Transformer architecture was introduced by Google researchers.")

# 3. Beispiel:
run_detailed_fact_check("Albert Einstein was a professional football player.")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUERY: Barack Obama was born in Kenya.
------------------------------------------------------------
AI is analyzing locally with Finetuned Intelligence...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



DETAILED ASSESSMENT:
Verdict: REFUTES

Reasoning:
The evidence provided clearly contradicts the statement that "Barack Obama was born in Kenya." The highest match score for this claim is 0.9431, which indicates a very strong match to the correct information that Barack Obama was actually born in Honolulu, Hawaii, United States. This high match score of 94.31% strongly supports the refutation of the given statement.

While the other pieces of evidence do not directly address the birthplace of Barack Obama, they provide additional context that further refutes the claim:
- The evidence stating that Barack Obama was the 44th President of the United States has a match score of 0.6076, indicating a moderate level of confidence in this piece of information. However, it does not directly address his place of birth.
- The evidence about Michelle Obama being born in the United States and more specifically in Boston, Massachusetts, has a match score of 0.6155, which also supports the general con

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



DETAILED ASSESSMENT:
Verdict: NOT ENOUGH INFO

Reasoning:
The provided evidence does not directly support or refute the claim that "The Transformer architecture was introduced by Google researchers." Let's break down the evidence and its match scores:

1. **Evidence: Transformers was promoted by a company. (Match Score: 0.5761)**
   - This evidence talks about the promotion of a product called Transformers, which could be related to a toy line or a movie franchise. It does not provide any information about the introduction of the Transformer architecture in machine learning.

2. **Evidence: Transformers used viral marketing for advertisement. (Match Score: 0.5747)**
   - Similar to the first piece of evidence, this statement discusses marketing strategies for a product called Transformers. Again, it does not provide any information about the origin of the Transformer architecture in the context of machine learning.

3. **Evidence: Transformers was a 2009 film. (Match Score: 0.5650)**


### Modelltraining und initiale Evaluierung

In [ ]:
import os
import torch
import pandas as pd
import gc
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, util
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss

# SETUP & KONFIGURATION
base_path = '/content/drive/MyDrive/BachlorArbeit'
model_id = "BAAI/bge-base-en-v1.5"
output_model_path = os.path.join(base_path, "models/bge_final_ba_pro")
log_dir = os.path.join(base_path, "tensorboard_logs")
data_path = os.path.join(base_path, "negatives_base_data.csv")

# BGE-Spezifische Instruktion
query_instruction = "Represent this sentence for searching relevant passages: "

# Ordner erstellen, falls nicht vorhanden
os.makedirs(os.path.dirname(output_model_path), exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# DATEN VORBEREITEN
print("Lade und transformiere Daten...")
df = pd.read_csv(data_path).dropna()
df['query_with_instruction'] = query_instruction + df['query'].astype(str)
temp_csv = os.path.join(base_path, "temp_bge_pro.csv")
df.to_csv(temp_csv, index=False)

# MINING & TRAINING
model = SentenceTransformer(model_id).cuda()
raw_dataset = load_dataset("csv", data_files=temp_csv, split="train")

print("Starte Hard-Negative Mining...")
dataset_with_negatives = util.mine_hard_negatives(
    dataset=raw_dataset,
    anchor_column_name="query_with_instruction",
    positive_column_name="positive",
    model=model,
    num_negatives=3,
    batch_size=128
)

train_test_split = dataset_with_negatives.train_test_split(test_size=0.1)

# Matryoshka Setup
inner_loss = MultipleNegativesRankingLoss(model=model)
train_loss = MatryoshkaLoss(
    model=model,
    loss=inner_loss,
    matryoshka_dims=[64, 128, 256, 512, 768]
)

# Training Arguments
training_args = SentenceTransformerTrainingArguments(
    output_dir=os.path.join(base_path, "bge_pro_checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=1e-5,
    bf16=True,
    save_total_limit=1,
    report_to="tensorboard",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_test_split["train"],
    eval_dataset=train_test_split["test"],
    loss=train_loss
)

print("Starte Training...")
trainer.train()
model.save(output_model_path)

# Speicher bereinigen
del model
gc.collect()
torch.cuda.empty_cache()

print("\nStarte Evaluierung...")

# Test-Stichprobe vorbereiten
test_sample = df.sample(min(250, len(df)))
queries = {f"q_{i}": row.query_with_instruction for i, row in enumerate(test_sample.itertuples())}
corpus = {f"d_{j}": text for j, text in enumerate(df['positive'].unique())}

# Relevanten Docs Map erstellen
text_to_id = {text: f"d_{j}" for j, text in enumerate(df['positive'].unique())}
relevant_docs = {f"q_{i}": {text_to_id[row.positive]} for i, row in enumerate(test_sample.itertuples())}

evaluator = InformationRetrievalEvaluator(queries, corpus, relevant_docs, name="FactCheck_Final")

def get_scores(m_id):
    print(f"Evaluiere Modell: {m_id}")
    m = SentenceTransformer(m_id).cuda()
    res = evaluator(m)

    # Hilfsfunktion zum sicheren Auslesen der Keys (mit oder ohne Prefix)
    def fetch(metric):
        return res.get(f"FactCheck_Final_{metric}", res.get(metric, 0.0))

    s1 = fetch("cosine_accuracy@1")
    s3 = fetch("cosine_accuracy@3")
    s5 = fetch("cosine_accuracy@5")
    s10 = fetch("cosine_accuracy@10")
    mrr = fetch("cosine_mrr@10")
    return [s1, s3, s5, s10, mrr]

res_base = get_scores(model_id)
res_pro = get_scores(output_model_path)

# Resultate-Tabelle
comparison_df = pd.DataFrame({
    "Metrik": ["S@1", "S@3", "S@5", "S@10", "MRR"],
    "Basis_BGE": res_base,
    "Finetuned_PRO": res_pro
})

# Speichern
comparison_df.to_csv(os.path.join(base_path, "final_evaluation_results.csv"), index=False)

print("\n" + "="*50)
print("PROJEKT ABGESCHLOSSEN")
print("="*50)
print(comparison_df.to_string(index=False))

Lade und transformiere Daten...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Starte Hard-Negative Mining...
Setting range_max to 12 based on the provided parameters.
Found 2509 unique queries out of 3401 total queries.
Found an average of 1.356 positives per query.


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

...


Starte finale Evaluierung...
Evaluiere Modell: BAAI/bge-base-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluiere Modell: /content/drive/MyDrive/BachlorArbeit/models/bge_final_ba_pro


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


PROJEKT ABGESCHLOSSEN
Metrik  Basis_BGE  Finetuned_PRO
   S@1    0.48000       0.620000
   S@3    0.72400       0.896000
   S@5    0.78000       0.960000
  S@10    0.81200       0.972000
   MRR    0.60686       0.762638


### Verfeinertes Modelltraining und Holdout-Evaluierung

In [ ]:
import os
import torch
import pandas as pd
import gc
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, util
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.losses import MultipleNegativesRankingLoss, MatryoshkaLoss

# SETUP & KONFIGURATION
base_path = '/content/drive/MyDrive/BachlorArbeit'
model_id = "BAAI/bge-base-en-v1.5"
output_model_path = os.path.join(base_path, "models/bge_final_ba_pro")
log_dir = os.path.join(base_path, "tensorboard_logs")
data_path = os.path.join(base_path, "negatives_base_data.csv")

# BGE-Spezifische Instruktion
query_instruction = "Represent this sentence for searching relevant passages: "

# Ordner erstellen, falls nicht vorhanden
os.makedirs(os.path.dirname(output_model_path), exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

# DATEN VORBEREITEN
print("Lade und transformiere Daten...")
df = pd.read_csv(data_path).dropna()
df['query_with_instruction'] = query_instruction + df['query'].astype(str)
temp_csv = os.path.join(base_path, "temp_bge_pro.csv")
df.to_csv(temp_csv, index=False)

# MINING & TRAINING (MIT TENSORBOARD)
model = SentenceTransformer(model_id).cuda()
raw_dataset = load_dataset("csv", data_files=temp_csv, split="train")

print("Starte Hard-Negative Mining...")
dataset_with_negatives = util.mine_hard_negatives(
    dataset=raw_dataset,
    anchor_column_name="query_with_instruction",
    positive_column_name="positive",
    model=model,
    num_negatives=3,
    batch_size=128
)

train_test_split = dataset_with_negatives.train_test_split(test_size=0.1)

# Matryoshka Setup
inner_loss = MultipleNegativesRankingLoss(model=model)
train_loss = MatryoshkaLoss(
    model=model,
    loss=inner_loss,
    matryoshka_dims=[64, 128, 256, 512, 768]
)

# Training Arguments
training_args = SentenceTransformerTrainingArguments(
    output_dir=os.path.join(base_path, "bge_pro_checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=1e-5,
    bf16=True,
    save_total_limit=1,
    report_to="tensorboard",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_test_split["train"],
    eval_dataset=train_test_split["test"],
    loss=train_loss
)

print("Starte Training...")
trainer.train()
model.save(output_model_path)

# Speicher bereinigen
del model
gc.collect()
torch.cuda.empty_cache()

# 4. FINALE EVALUIERUNG (BASIS VS. FINETUNED)
print("\nStarte finale Evaluierung...")

test_ds = train_test_split["test"]
test_sample_df = test_ds.to_pandas()
train_ds_df = train_test_split["train"].to_pandas()

queries = {
    f"q_{i}": row.query_with_instruction
    for i, row in enumerate(test_sample_df.itertuples())
}

corpus_texts = pd.concat([
    train_ds_df['positive'],
    test_sample_df['positive']
]).unique()

corpus = {f"d_{j}": text for j, text in enumerate(corpus_texts)}
text_to_id = {text: f"d_{j}" for j, text in enumerate(corpus_texts)}

relevant_docs = {
    f"q_{i}": {text_to_id[row.positive]}
    for i, row in enumerate(test_sample_df.itertuples())
}

evaluator = InformationRetrievalEvaluator(queries, corpus, relevant_docs, name="FactCheck_Final")

def get_scores(m_id):
    print(f"Evaluiere Modell: {m_id}")
    m = SentenceTransformer(m_id).cuda()

    eval_result = evaluator(m)
    res = eval_result[1] if isinstance(eval_result, tuple) else eval_result

    s1 = res.get("FactCheck_Final_cosine_accuracy@1", 0.0)
    s3 = res.get("FactCheck_Final_cosine_accuracy@3", 0.0)
    s5 = res.get("FactCheck_Final_cosine_accuracy@5", 0.0)
    s10 = res.get("FactCheck_Final_cosine_accuracy@10", 0.0)
    mrr = res.get("FactCheck_Final_cosine_mrr@10", 0.0)
    return [s1, s3, s5, s10, mrr]

res_base = get_scores(model_id)
res_pro = get_scores(output_model_path)

comparison_df = pd.DataFrame({
    "Metrik": ["S@1", "S@3", "S@5", "S@10", "MRR"],
    "Basis_BGE": res_base,
    "Finetuned_PRO": res_pro
})

comparison_df.to_csv(os.path.join(base_path, "final_evaluation_results_holdout.csv"), index=False)

print("\n" + "="*50)
print("PROJEKT ABGESCHLOSSEN (Evaluation auf echtem Holdout)")
print("="*50)
print(comparison_df.to_string(index=False))

# EVALUIERUNG (BASIS VS. FINETUNED)

print("\nStarte finale Evaluierung...")

test_ds = train_test_split["test"]
test_sample_df = test_ds.to_pandas()

train_ds_df = train_test_split["train"].to_pandas()

queries = {
    f"q_{i}": row.query_with_instruction
    for i, row in enumerate(test_sample_df.itertuples())
}

# Korpus: Trainings-Positives + Test-Positives zusammenführen
corpus_texts = pd.concat([
    train_ds_df['positive'],
    test_sample_df['positive']
]).unique()

corpus = {f"d_{j}": text for j, text in enumerate(corpus_texts)}
text_to_id = {text: f"d_{j}" for j, text in enumerate(corpus_texts)}

# Relevante Docs Map erstellen (nur für die Test-Queries)
relevant_docs = {
    f"q_{i}": {text_to_id[row.positive]}
    for i, row in enumerate(test_sample_df.itertuples())
}

evaluator = InformationRetrievalEvaluator(queries, corpus, relevant_docs, name="FactCheck_Final")

def get_scores(m_id):
    print(f"Evaluiere Modell: {m_id}")
    m = SentenceTransformer(m_id).cuda()
    res = evaluator(m)
    # Metriken extrahieren
    s1 = res.get("FactCheck_Final_cosine_accuracy@1", 0.0)
    s3 = res.get("FactCheck_Final_cosine_accuracy@3", 0.0)
    s5 = res.get("FactCheck_Final_cosine_accuracy@5", 0.0)
    s10 = res.get("FactCheck_Final_cosine_accuracy@10", 0.0)
    mrr = res.get("FactCheck_Final_cosine_mrr@10", 0.0)
    return [s1, s3, s5, s10, mrr]

res_base = get_scores(model_id)
res_pro = get_scores(output_model_path)

# Resultate-Tabelle
comparison_df = pd.DataFrame({
    "Metrik": ["S@1", "S@3", "S@5", "S@10", "MRR"],
    "Basis_BGE": res_base,
    "Finetuned_PRO": res_pro
})

# Speichern
comparison_df.to_csv(os.path.join(base_path, "final_evaluation_results_holdout.csv"), index=False)

print("\n" + "="*50)
print("PROJEKT ABGESCHLOSSEN (Evaluation auf echtem Holdout)")
print("="*50)
print(comparison_df.to_string(index=False))

Lade und transformiere Daten...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Starte Hard-Negative Mining...
Setting range_max to 12 based on the provided parameters.
Found 2509 unique queries out of 3401 total queries.
Found an average of 1.356 positives per query.


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

...


PROJEKT ABGESCHLOSSEN (Evaluation auf echtem Holdout)
Metrik  Basis_BGE  Finetuned_PRO
   S@1   0.532359       0.655532
   S@3   0.711900       0.879958
   S@5   0.754697       0.920668
  S@10   0.790188       0.941545
   MRR   0.628976       0.769316

Starte finale Evaluierung...
Evaluiere Modell: BAAI/bge-base-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluiere Modell: /content/drive/MyDrive/BachlorArbeit/models/bge_final_ba_pro


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


PROJEKT ABGESCHLOSSEN (Evaluation auf echtem Holdout)
Metrik  Basis_BGE  Finetuned_PRO
   S@1   0.532359       0.655532
   S@3   0.711900       0.879958
   S@5   0.754697       0.920668
  S@10   0.790188       0.941545
   MRR   0.628976       0.769316


### Data-Leakage-Prüfung und Anzeige der Evaluationsergebnisse

In [ ]:
import json
import os
import pandas as pd

base_path = "/content/drive/MyDrive/BachlorArbeit/"

train_file_path = os.path.join(base_path, "negatives_base_data.csv")
fever_corpus_path = os.path.join(base_path, "train.jsonl")
eval_results_path = os.path.join(base_path, "final_evaluation_results.csv")


def load_claims_from_file(file_path):
    """Lädt Text-Claims aus .json, .jsonl oder .csv Dateien."""
    if not os.path.exists(file_path):
        return set(), 0

    claims = []

    try:
        if file_path.endswith(".json"):
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, list):
                    for item in data:
                        if isinstance(item, dict):
                            claim = (
                                item.get("query")
                                or item.get("claim")
                                or item.get("text")
                                or item.get("question")
                            )
                            if claim:
                                claims.append(str(claim).strip())
                        elif isinstance(item, str):
                            claims.append(item.strip())
                elif isinstance(data, dict):
                    for val in data.values():
                        if isinstance(val, list):
                            claims.extend(
                                [str(x).strip() for x in val if isinstance(x, str)]
                            )

        elif file_path.endswith(".jsonl"):
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        claim = data.get("claim") or data.get("query")
                        if claim:
                            claims.append(str(claim).strip())
                    except json.JSONDecodeError:
                        continue

        elif file_path.endswith(".csv"):
            df = pd.read_csv(file_path)
            target_col = None
            for col in [
                "query",
                "claim",
                "positive",
                "english_text",
                "question",
            ]:
                if col in df.columns:
                    target_col = col
                    break
            if target_col:
                claims = (
                    df[target_col].dropna().astype(str).str.strip().tolist()
                )
            else:
                claims = df.iloc[:, 0].dropna().astype(str).str.strip().tolist()
    except Exception as e:
        print(f"Fehler beim Lesen von {os.path.basename(file_path)}: {e}")
        return set(), 0

    return set(claims), len(claims)


print("=" * 65)
print("1. SUCHE NACH DEM EVALUATIONSET IN UNTERORDNERN (test_set / train_dev_sets)")
print("=" * 65)

found_eval_path = None

# Ordner durchsuchen: Hauptpfad, test_set und train_dev_sets
search_dirs = [
    base_path,
    os.path.join(base_path, "test_set"),
    os.path.join(base_path, "train_dev_sets"),
]

for s_dir in search_dirs:
    if os.path.exists(s_dir):
        for fname in os.listdir(s_dir):
            full_p = os.path.join(s_dir, fname)
            if os.path.isfile(full_p) and fname.endswith(
                (".json", ".jsonl", ".csv")
            ):
                if fname in ["train.jsonl", "negatives_base_data.csv"]:
                    continue
                c_set, c_len = load_claims_from_file(full_p)
                rel_path = os.path.relpath(full_p, base_path)
                print(
                    f"Datei: {rel_path:<40} | Einträge: {c_len:<5} | Eindeutig: {len(c_set)}"
                )

                # Identifikation des Test-Sets (~500 Einträge oder Namens-Match)
                if (
                    (450 <= c_len <= 550)
                    or ("eval" in fname.lower())
                    or ("test" in fname.lower())
                ) and not found_eval_path:
                    found_eval_path = full_p

if found_eval_path:
    print(
        f"\n--> Gewähltes Eval-Set: '{os.path.relpath(found_eval_path, base_path)}'"
    )
else:
    print("\n Keines der Sets konnte automatisch ausgewählt werden.")

print("\n" + "=" * 65)
print("2. DATA LEAKAGE PRÜFUNG (Punkte G7, G8, 75, 78)")
print("=" * 65)

train_claims, train_count = load_claims_from_file(train_file_path)
fever_claims, fever_count = load_claims_from_file(fever_corpus_path)

if found_eval_path:
    eval_claims, eval_count = load_claims_from_file(found_eval_path)

    print("--- DATASET OVERVIEW ---")
    print(
        f"1. Training Set (negatives_base_data.csv): {train_count} Einträge | {len(train_claims)} eindeutige Claims"
    )
    print(
        f"2. FEVER Corpus (train.jsonl):              {fever_count} Einträge | {len(fever_claims)} eindeutige Claims"
    )
    print(
        f"3. Evaluation Set ({os.path.basename(found_eval_path)}):      {eval_count} Einträge | {len(eval_claims)} eindeutige Claims\n"
    )

    leakage = eval_claims.intersection(train_claims)
    print("--- DATA LEAKAGE ERGEBNIS ---")
    if len(leakage) == 0:
        print(
            " PERFEKT: 0 Evaluierungs-Claims wurden im Trainings-Set gefunden!"
        )
        print(
            "\nFormulierung für Ihre Bachelorarbeit:"
        )
        print(
            ' "Die 500 Evaluierungs-Claims wurden weder im Training noch beim Hard-Negative Mining verwendet (Data Leakage Exclusion)."'
        )
    else:
        print(
            f" WARNUNG: {len(leakage)} Überschneidungen zwischen Eval-Set und Train-Set gefunden!"
        )
        for sample in list(leakage)[:3]:
            print(f" - {sample}")

print("\n" + "=" * 65)
print("3. INHALT VON final_evaluation_results.csv")
print("=" * 65)
if os.path.exists(eval_results_path):
    df_res = pd.read_csv(eval_results_path)
    print(df_res.to_string(index=False))
else:
    print("Datei final_evaluation_results.csv nicht gefunden.")
print("=" * 65)

1. SUCHE NACH DEM EVALUATIONSET IN UNTERORDNERN (test_set / train_dev_sets)
Datei: pairs.csv                                | Einträge: 25743 | Eindeutig: 21988
Datei: eng_posts.csv                            | Einträge: 4022  | Eindeutig: 3725
Datei: eng_fact.csv                             | Einträge: 85734 | Eindeutig: 85463
Datei: eng_pairs_cleaned.csv                    | Einträge: 3401  | Eindeutig: 2675
Datei: training_data_final.csv                  | Einträge: 3401  | Eindeutig: 2487
Datei: bge_train_data.csv                       | Einträge: 3401  | Eindeutig: 2509
Datei: temp_bge_train.csv                       | Einträge: 3401  | Eindeutig: 2491
Datei: temp_bge_final.csv                       | Einträge: 3401  | Eindeutig: 2509
Datei: evaluation_results.csv                   | Einträge: 5     | Eindeutig: 5
Datei: final_evaluation_results.csv             | Einträge: 5     | Eindeutig: 5
Datei: temp_bge_pro.csv                         | Einträge: 3401  | Eindeutig: 2509
Date

### End-to-End RAG Pipeline Evaluierung (Kapitel 6.3)

In [ ]:
import os
import json
import torch
import re
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sklearn.metrics import accuracy_score, recall_score


# KONFIGURATION & PFADE
BASE_PATH = "/content/drive/MyDrive/BachlorArbeit/"
FILE_PATH = os.path.join(BASE_PATH, "train.jsonl")
MODEL_BASE_NAME = "BAAI/bge-base-en-v1.5"
EMBEDDINGS_BASE_PATH = os.path.join(BASE_PATH, "fever_embeddings_bge_base.pt")
MODEL_PRO_PATH = os.path.join(BASE_PATH, "models/bge_final_ba_pro")
EMBEDDINGS_PRO_PATH = os.path.join(BASE_PATH, "fever_embeddings_bge_pro.pt")

QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "
LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# LLM & DATEN INITIALISIEREN
print("--- Loading Local Qwen Language Model ---")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
local_generator = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer
)

# FEVER-Daten laden
data = []
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))
df_fever = pd.DataFrame(data)

# Basis-Modell & Embeddings
print("\n--- Initializing Base Retrieval System ---")
model_base = SentenceTransformer(MODEL_BASE_NAME).cuda()
if os.path.exists(EMBEDDINGS_BASE_PATH):
    emb_base = torch.load(EMBEDDINGS_BASE_PATH)
else:
    emb_base = model_base.encode(df_fever['claim'].tolist(), convert_to_tensor=True, show_progress_bar=True)
    torch.save(emb_base, EMBEDDINGS_BASE_PATH)

# Pro-Modell (Finetuned) & Embeddings
print("\n--- Initializing Finetuned (Pro) Retrieval System ---")
model_pro = SentenceTransformer(MODEL_PRO_PATH).cuda()
if os.path.exists(EMBEDDINGS_PRO_PATH):
    emb_pro = torch.load(EMBEDDINGS_PRO_PATH)
else:
    emb_pro = model_pro.encode(df_fever['claim'].tolist(), convert_to_tensor=True, show_progress_bar=True)
    torch.save(emb_pro, EMBEDDINGS_PRO_PATH)

# BALANCIERTES TEST-SET & BATCH-RETRIEVAL

valid_labels = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]
df_filtered = df_fever[df_fever['label'].isin(valid_labels)].copy()

# (Zum schnellen Testen kannst man hier auch 20 statt 100 nehmen)
TEST_N_PER_CLASS = 100

df_gold = df_filtered.groupby("label", group_keys=False).apply(
    lambda x: x.sample(n=min(TEST_N_PER_CLASS, len(x)), random_state=42)
).reset_index(drop=True)

print(f"\nBalanciertes Test-Set erstellt mit insgesamt {len(df_gold)} Beispielen.")

print("\nFühre GPU-Batch Retrieval für alle Queries parallel aus...")
full_queries = [QUERY_INSTRUCTION + q for q in df_gold['claim'].tolist()]

query_embs_base = model_base.encode(full_queries, convert_to_tensor=True, batch_size=64)
query_embs_pro = model_pro.encode(full_queries, convert_to_tensor=True, batch_size=64)

hits_base_all = util.semantic_search(query_embs_base, emb_base, top_k=5)
hits_pro_all = util.semantic_search(query_embs_pro, emb_pro, top_k=5)

# INFERENZ-LOGIK
def get_verdict_safe(query, hits):
    evidence_context = ""
    for hit in hits:
        if hit['score'] >= 0.30:
            idx = hit['corpus_id']
            row = df_fever.iloc[idx]
            evidence_context += f"- Evidence Text: {row['claim']} (Match Score: {hit['score']:.4f})\n"

    messages = [
        {"role": "system", "content": "You are a professional fact-checker. Analyze whether the evidence supports, refutes, or provides not enough info for the user's claim. Provide detailed English responses with logical reasoning."},
        {"role": "user", "content": f"Here is the retrieved evidence from the database:\n{evidence_context}\nUser Claim to verify: {query}\nTask:\n1. Provide a verdict: SUPPORTS, REFUTES, or NOT ENOUGH INFO.\n2. Explain your logical reasoning in English based strictly on the provided evidence."}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = local_generator(
        prompt,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        return_full_text=False
    )

    response = outputs[0]["generated_text"]
    verdict_match = re.search(r'\b(SUPPORTS|REFUTES|NOT ENOUGH INFO)\b', response, re.IGNORECASE)
    if verdict_match:
        return verdict_match.group(1).upper()
    return "NOT ENOUGH INFO"

# DURCHLAUF & EVALUATIONSTABELLE
print("\nStarte Inferenz-Durchlauf für Base & Pro (kann je nach Anzahl etwas dauern)...")
preds_base = []
preds_pro = []

for idx in tqdm(range(len(df_gold)), desc="LLM Verification"):
    claim = df_gold.iloc[idx]['claim']
    preds_base.append(get_verdict_safe(claim, hits_base_all[idx]))
    preds_pro.append(get_verdict_safe(claim, hits_pro_all[idx]))

df_gold['pred_base'] = preds_base
df_gold['pred_pro'] = preds_pro

# Metriken nach sklearn Logik berechnen (Precision / Accuracy / Error Rate)
from sklearn.metrics import precision_score

# Gesamt-Accuracy
acc_base = accuracy_score(df_gold["label"], df_gold["pred_base"])
acc_pro = accuracy_score(df_gold["label"], df_gold["pred_pro"])

# Klassen-spezifische Precision (Sicherer Umgang mit Zero Division)
prec_base = precision_score(df_gold["label"], df_gold["pred_base"], labels=["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"], average=None, zero_division=0)
prec_pro = precision_score(df_gold["label"], df_gold["pred_pro"], labels=["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"], average=None, zero_division=0)

# Anteil an Halluzinationen / Fehlurteilen (1.0 - Accuracy)
error_base = 1.0 - acc_base
error_pro = 1.0 - acc_pro

# Erstellung des DataFrames
eval_summary = pd.DataFrame({
    "Evaluierte Metrik des LLM-Verdicts": [
        "Gesamt-Genauigkeit (Accuracy)",
        "Präzision (Precision) -- SUPPORTS",
        "Präzision (Precision) -- REFUTES",
        "Anteil an Halluzinationen / Fehlurteilen"
    ],
    "RAG mit BGE-Base (Untrainiert)": [
        f"{acc_base*100:.1f} %",
        f"{prec_base[0]*100:.1f} %",
        f"{prec_base[1]*100:.1f} %",
        f"{error_base*100:.1f} %"
    ],
    "RAG mit BGE-Pro (Gefinetunt)": [
        f"{acc_pro*100:.1f} %",
        f"{prec_pro[0]*100:.1f} %",
        f"{prec_pro[1]*100:.1f} %",
        f"{error_pro*100:.1f} %"
    ],
    "Absolute Veränderung": [
        f"{(acc_pro - acc_base)*100:+.1f} Prozentpunkte",
        f"{(prec_pro[0] - prec_base[0])*100:+.1f} Prozentpunkte",
        f"{(prec_pro[1] - prec_base[1])*100:+.1f} Prozentpunkte",
        f"{(error_pro - error_base)*100:+.1f} Prozentpunkte"
    ]
})

print("\n" + "="*100)
print(f"TABELLE 6.3: End-to-End-Klassifikationsgenauigkeit (Basierend auf {len(df_gold)} Test-Beispielen)")
print("="*100)
print(eval_summary.to_string(index=False))

--- Loading Local Qwen Language Model ---


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


--- Initializing Base Retrieval System ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


--- Initializing Finetuned (Pro) Retrieval System ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Balanciertes Test-Set erstellt mit insgesamt 300 Beispielen.

Führe GPU-Batch Retrieval für alle Queries parallel aus...

Starte Inferenz-Durchlauf für Base & Pro (kann je nach Anzahl etwas dauern)...


LLM Verification:   0%|          | 0/300 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
LLM Verification:   0%|          | 1/300 [00:07<34:57,  7.02s/it][transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been se


TABELLE 6.3: End-to-End-Klassifikationsgenauigkeit (Basierend auf 300 Test-Beispielen)
      Evaluierte Metrik des LLM-Verdicts RAG mit BGE-Base (Untrainiert) RAG mit BGE-Pro (Gefinetunt) Absolute Veränderung
           Gesamt-Genauigkeit (Accuracy)                         57.3 %                       58.3 %   +1.0 Prozentpunkte
       Präzision (Precision) -- SUPPORTS                         50.8 %                       51.9 %   +1.1 Prozentpunkte
        Präzision (Precision) -- REFUTES                         66.4 %                       67.9 %   +1.5 Prozentpunkte
Anteil an Halluzinationen / Fehlurteilen                         42.7 %                       41.7 %   -1.0 Prozentpunkte


### Detaillierte LLM-Urteils-Evaluierungstabelle

In [ ]:
from sklearn.metrics import accuracy_score, precision_score
import pandas as pd

labels = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]

# PRÄZISION PRO KLASSE
prec_base = precision_score(
    df_gold["label"],
    df_gold["pred_base"],
    labels=labels,
    average=None,
    zero_division=0
)

prec_pro = precision_score(
    df_gold["label"],
    df_gold["pred_pro"],
    labels=labels,
    average=None,
    zero_division=0
)

# GESAMTGENAUIGKEIT & HALLUZINATIONSQUOTE
acc_base = accuracy_score(df_gold["label"], df_gold["pred_base"])
acc_pro = accuracy_score(df_gold["label"], df_gold["pred_pro"])

hall_base = 1.0 - acc_base
hall_pro = 1.0 - acc_pro

total_examples = len(df_gold)
correct_base_total = int((df_gold['label'] == df_gold['pred_base']).sum())
correct_pro_total = int((df_gold['label'] == df_gold['pred_pro']).sum())

# ERWEITERTE EVALUATIONSTABELLE
from sklearn.metrics import accuracy_score, precision_score, recall_score
import pandas as pd

labels = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]

# Metriken berechnen
acc_base = accuracy_score(df_gold["label"], df_gold["pred_base"])
acc_pro = accuracy_score(df_gold["label"], df_gold["pred_pro"])

prec_base = precision_score(df_gold["label"], df_gold["pred_base"], labels=labels, average=None, zero_division=0)
prec_pro = precision_score(df_gold["label"], df_gold["pred_pro"], labels=labels, average=None, zero_division=0)

hall_base = 1.0 - acc_base
hall_pro = 1.0 - acc_pro

total_examples = len(df_gold)
correct_base_total = int((df_gold['label'] == df_gold['pred_base']).sum())
correct_pro_total = int((df_gold['label'] == df_gold['pred_pro']).sum())

# Erstellung der vollständigen DataFrame-Tabelle
eval_summary = pd.DataFrame({
    "Evaluierte Metrik des LLM-Verdicts": [
        "Gesamt-Genauigkeit (Accuracy)",
        "Präzision (Precision) -- SUPPORTS",
        "Präzision (Precision) -- REFUTES",
        "Präzision (Precision) -- NOT ENOUGH INFO",
        "Anteil an Halluzinationen / Fehlurteilen"
    ],
    "RAG mit BGE-Base (Untrainiert)": [
        f"{acc_base * 100:.1f} % ({correct_base_total}/{total_examples})",
        f"{prec_base[0] * 100:.1f} %",
        f"{prec_base[1] * 100:.1f} %",
        f"{prec_base[2] * 100:.1f} %",
        f"{hall_base * 100:.1f} %"
    ],
    "RAG mit BGE-Pro (Gefinetunt)": [
        f"{acc_pro * 100:.1f} % ({correct_pro_total}/{total_examples})",
        f"{prec_pro[0] * 100:.1f} %",
        f"{prec_pro[1] * 100:.1f} %",
        f"{prec_pro[2] * 100:.1f} %",
        f"{hall_pro * 100:.1f} %"
    ],
    "Absolute Veränderung": [
        f"{(acc_pro - acc_base) * 100:+.1f} Prozentpunkte",
        f"{(prec_pro[0] - prec_base[0]) * 100:+.1f} Prozentpunkte",
        f"{(prec_pro[1] - prec_base[1]) * 100:+.1f} Prozentpunkte",
        f"{(prec_pro[2] - prec_base[2]) * 100:+.1f} Prozentpunkte",
        f"{(hall_pro - hall_base) * 100:+.1f} Prozentpunkte"
    ]
})

print("\n" + "=" * 115)
print(f" VOLLSTÄNDIGE EVALUATIONSTABELLE (Basierend auf {total_examples} Test-Beispielen)")
print("=" * 115)
print(eval_summary.to_string(index=False))

# CSV-Export für LaTeX / Backup
eval_summary.to_csv(
    "evaluation_chapter_6_3_complete.csv",
    index=False,
    encoding="utf-8-sig"
)


 VOLLSTÄNDIGE EVALUATIONSTABELLE (Basierend auf 300 Test-Beispielen)
      Evaluierte Metrik des LLM-Verdicts RAG mit BGE-Base (Untrainiert) RAG mit BGE-Pro (Gefinetunt) Absolute Veränderung
           Gesamt-Genauigkeit (Accuracy)               57.3 % (172/300)             58.3 % (175/300)   +1.0 Prozentpunkte
       Präzision (Precision) -- SUPPORTS                         50.8 %                       51.9 %   +1.1 Prozentpunkte
        Präzision (Precision) -- REFUTES                         66.4 %                       67.9 %   +1.5 Prozentpunkte
Präzision (Precision) -- NOT ENOUGH INFO                         77.8 %                       71.4 %   -6.3 Prozentpunkte
Anteil an Halluzinationen / Fehlurteilen                         42.7 %                       41.7 %   -1.0 Prozentpunkte
